In [45]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr

In [46]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


Google API Key exists and begins AI
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


In [47]:

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

In [48]:

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [49]:
def generate_content(tool, text):

    if tool == "Tweet Generator":
        prompt = f"Generate a tweet under 280 characters about: {text}"

    elif tool == "Hashtag Generator":
        prompt = f"Generate 10 trending hashtags for the topic: {text}"

    elif tool == "Caption Generator":
        prompt = f"Generate an engaging Instagram caption about: {text}"

    elif tool == "Story Generator":
        prompt = f"Write a short creative story about: {text} under 150 words"

    elif tool == "Text Summarizer":
        prompt = f"Summarize the following text in 3 sentences:\n{text}"


    stream = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": "You are a helpful AI content generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        stream=True
    )

    response = ""

    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        response += content
        yield response

In [50]:
tool = gr.Dropdown(
    [
        "Tweet Generator",
        "Hashtag Generator",
        "Caption Generator",
        "Story Generator",
        "Text Summarizer"
    ],
    label="Choose Tool"
)

user_input = gr.Textbox(
    label="Enter Topic or Text",
    lines=5
)

output = gr.Textbox(
    label="Generated Content",
    lines=10
)

interface = gr.Interface(
    fn=generate_content,
    inputs=[tool, user_input],
    outputs=output,
    title="AI Content Toolkit using LLMs",
    description="Generate tweets, hashtags, captions, stories, and summaries using LLMs"
)

In [55]:
with gr.Blocks(title="ContentForge AI") as interface:
    gr.Markdown("""# ContentForge AI
A simple content studio to generate tweets, hashtags, captions, short stories, and summaries from a single prompt.
""")

    with gr.Row():
        tool = gr.Dropdown(
            [
                "Tweet Generator",
                "Hashtag Generator",
                "Caption Generator",
                "Story Generator",
                "Text Summarizer",
            ],
            label="Content Type",
            value="Tweet Generator",
            info="What kind of content would you like to generate?",
        )

    with gr.Row():
        with gr.Column(scale=1):
            user_input = gr.Textbox(
                label="Topic or Text",
                placeholder="e.g. Launching a new AI-powered productivity app",
                lines=8,
            )
            generate_button = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            output = gr.Textbox(
                label="Generated Content",
                lines=12,
            )

    gr.Markdown(
        """---
**Tip:** Refine your topic or text and click **Generate** again for alternative versions.
"""
    )

    generate_button.click(fn=generate_content, inputs=[tool, user_input], outputs=output)

In [56]:
interface.launch(share='true')

* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://555fb67815f025eeec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
